# Chapter 9 — When the Metric Becomes the Target

**Book alignment:** DSPy From First Principles, Chapter 9

**Question this notebook isolates:** Can a candidate that reverses the author's meaning score highly under v1 while satisfying every implemented check?

In [ ]:
from pathlib import Path
import random
import sys

random.seed(13)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy  # imported for construction only; no LM is ever called

from ch09_metric_attack.run import ATTACK_ORDER, candidates
from common.data import canonical_split
from common.metrics import editorial_metric

## The gate catches the cheap attacks

Deterministic string-built attacks cost nothing and reproduce exactly. Required-entity renames and empty outputs must fail the hard gate and score zero; the exact reference must pass.

In [ ]:
split = canonical_split()
by_id = {c.case_id: c for c in list(split.train) + list(split.dev)}

gate_rows = []
for case_id in ("ed-001", "ed-002", "ed-003"):
    case = by_id[case_id]
    specs = candidates(case)
    assert tuple(specs) == ATTACK_ORDER
    for name in ("required_entity_rename", "empty_output", "copy_reference"):
        breakdown = editorial_metric(case, specs[name]["text"], version="v1")
        gate_rows.append({
            "case_id": case_id,
            "attack": name,
            "score": round(breakdown.score, 3),
            "gate": breakdown.hard_gate_passed,
        })

gate_rows

In [ ]:
for r in gate_rows:
    if r["attack"] in ("required_entity_rename", "empty_output"):
        assert r["score"] == 0.0 and r["gate"] is False, r
    else:
        assert r["gate"] is True, r

print("renames and empties gated to 0.0; references pass")

## The semantic flip walks straight through

Each flip takes the reference rewrite and reverses its meaning while changing as few words as possible. v1 sees shared vocabulary and rewards it; the book's v2 combination rule (replayed here with a recorded violated verdict, no judge called) caps it at 0.30.

In [ ]:
def v2_style_score(structural: float, n_violated: int) -> float:
    """Replay the book's v2 rule: structural * (0.30 + 0.70 * factor), cap 0.30 on violation."""
    if n_violated:
        return min(structural * 0.30, 0.30)
    return structural


flip_rows = []
for case_id in ("ed-001", "ed-002", "ed-003"):
    case = by_id[case_id]
    text = candidates(case)["semantic_flip_high_overlap"]["text"]
    v1 = editorial_metric(case, text, version="v1")
    flip_rows.append({
        "case_id": case_id,
        "flip": text,
        "v1": round(v1.score, 3),
        "gate": v1.hard_gate_passed,
        "unchecked": list(v1.semantic_constraints_unchecked),
        "v2_replay": round(v2_style_score(v1.score, len(case.semantic_constraints)), 3),
    })

[(r["case_id"], r["v1"], r["v2_replay"]) for r in flip_rows]

In [ ]:
for r in flip_rows:
    assert r["gate"] is True, r  # entities kept, no forbidden terms
    assert len(r["unchecked"]) > 0, r  # nothing in v1 consumes the constraints
    assert r["v1"] >= 0.84, r  # high-score region, near the ~0.79 baseline mean
    assert r["v2_replay"] <= 0.30, r

for r in flip_rows:
    print(f"{r['case_id']}: v1 {r['v1']:.3f} -> v2-replay {r['v2_replay']:.3f}  |  {r['flip']}")

## What we earned

The deterministic gate does its job on renames and empties, but meaning reversal with preserved vocabulary scores 0.85–0.96 under v1: a candidate can satisfy every implemented check while violating the property the task requires. The semantic constraints that would catch it sit in the fixture, consumed by nothing.

Notebook 10 / Chapter 10 freezes the other half of the experiment — the compile boundary — before any optimizer is allowed to pursue this objective.